# 103 - Plantilla de Dashboard — **Soluciones**

Ejemplo con **Gradio** que carga el `model.joblib` si existe y expone un predictor.

In [ ]:

import json
from pathlib import Path
import numpy as np

HAS_GR=False
try:
    import gradio as gr
    HAS_GR=True
except Exception as e:
    print("ℹ Gradio no disponible:", e)

# Carga perezosa del modelo
artefact = Path("artifacts/model.joblib")
scaler = model = None
if artefact.exists():
    import joblib
    bundle = joblib.load(artefact)
    scaler, model = bundle["scaler"], bundle["model"]
    print("Modelo cargado desde artifacts/model.joblib")
else:
    print("Sin modelo entrenado; se usará una regla dummy.")

def predictor(f1: float, f2: float) -> float:
    x = np.array([[f1, f2]], dtype="float32")
    if scaler is not None and model is not None:
        xs = scaler.transform(x)
        return float(model.predict(xs)[0])
    return float(1.0 if (f1+f2)>0 else 0.0)

if HAS_GR:
    demo = gr.Interface(
        fn=predictor,
        inputs=[gr.Slider(-3,3,0,label="feature1"), gr.Slider(-3,3,0,label="feature2")],
        outputs=gr.Number(label="Predicción"),
        title="Proyecto — Dashboard de Predicción",
        description="Carga automática del modelo si existe en ./artifacts."
    )
    print("Ejecuta demo.launch() para abrir el dashboard.")
